In [ ]:
import numpy as np
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [ ]:
X = ["great product", "best service", "love it", "excellent", "bad", "terrible", "waste", "poor"]
y = [1, 1, 1, 1, 0, 0, 0, 0]

In [ ]:
def preprocess(text):
    stop_words = set(stopwords.words('english'))
    return [w for w in word_tokenize(text.lower()) if w.isalnum() and w not in stop_words]

In [ ]:
def one_hot(doc, vocab):
    vec = np.zeros(len(vocab))
    for w in doc:
        if w in vocab:
            vec[vocab.index(w)] = 1
    return vec

In [ ]:
processed = [preprocess(s) for s in X]
vocab = list(set(w for d in processed for w in d))
X_train = np.array([one_hot(d, vocab) for d in processed])

In [ ]:
class DecisionTree:
    def __init__(self, max_depth=2):
        self.max_depth = max_depth
    def fit(self, X, y):
        self.tree = self._grow(X, y)
    def _gini(self, y):
        return 1 - sum((np.sum(y == c) / len(y))**2 for c in np.unique(y))
    def _split(self, X, y):
        best_gini, best_split = float('inf'), None
        for feat in range(X.shape[1]):
            for thr in np.unique(X[:, feat]):
                left = y[X[:, feat] <= thr]
                right = y[X[:, feat] > thr]
                gini = (len(left)*self._gini(left) + len(right)*self._gini(right)) / len(y)
                if gini < best_gini:
                    best_gini, best_split = gini, (feat, thr)
        return best_split
    def _grow(self, X, y, depth=0):
        if len(np.unique(y)) == 1 or depth >= self.max_depth:
            return np.bincount(y).argmax()
        feat, thr = self._split(X, y)
        left_idx = X[:, feat] <= thr
        return (feat, thr, self._grow(X[left_idx], y[left_idx], depth+1), self._grow(X[~left_idx], y[~left_idx], depth+1))
    def _predict_one(self, x, tree):
        if not isinstance(tree, tuple):
            return tree
        feat, thr, left, right = tree
        return self._predict_one(x, left) if x[feat] <= thr else self._predict_one(x, right)
    def predict(self, X):
        return [self._predict_one(x, self.tree) for x in X]

In [ ]:
class RandomForest:
    def __init__(self, n_trees=10, max_depth=2):
        self.trees = [DecisionTree(max_depth) for _ in range(n_trees)]
    def fit(self, X, y):
        for tree in self.trees:
            idx = np.random.choice(len(X), len(X), replace=True)
            tree.fit(X[idx], y[idx])
    def predict(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees]).T
        return [np.bincount(p).argmax() for p in predictions]

In [ ]:
model = RandomForest()
model.fit(X_train, y)

In [ ]:
test = one_hot(preprocess("great service"), vocab)
test_vec = np.array([test])
print("Prediction:", model.predict(test_vec)[0])